# MES LOT 품질 분석

분석단위는 `지시번호 × 작업순서 × 실적순번 × LOT NO`입니다. 공정 요약행과 상세 실적행은 다른 집계 수준으로 구분합니다. 입력 데이터·전사값·세부 설명은 공개 소스에 포함하지 않고 private 입력 경로의 `analysis_config.json`과 해당 파일에서 읽습니다.

기존 입력·인벤토리·원문 컬럼·결측/중복·키 매칭·수량 검산·시각화·저장 기능을 순서대로 실행합니다. 실제 원본은 읽기만 하고, 결과는 `MES_OUTPUT_DIR`의 새 실행 폴더에 저장합니다. 공개 배포본에는 실행 출력이 없습니다.

실제 데이터로 실행할 때는 이 Notebook을 private 작업폴더에 복사해 사용하세요. 공개 저장소에서 직접 실행·저장하면 셀 출력에 실제 데이터가 포함될 수 있습니다. `MES_DATA_DIR`은 private 입력 폴더, `MES_OUTPUT_DIR`은 private 출력 폴더, `MES_REPOSITORY_DIR`은 공개 저장소 위치입니다. 설정하지 않은 입력·출력은 홈 폴더의 `daegu_mes_private` 아래를 사용합니다. 입력과 출력은 공개 저장소 바깥이어야 합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from importlib import metadata
import hashlib, json, uuid, os
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

INPUT = Path(os.environ.get('MES_DATA_DIR', Path.home()/'daegu_mes_private'/'inputs')).expanduser().resolve()
OUTPUT = Path(os.environ.get('MES_OUTPUT_DIR', Path.home()/'daegu_mes_private'/'outputs')).expanduser().resolve()
PUBLIC_ROOT = Path(os.environ.get('MES_REPOSITORY_DIR', Path.cwd())).expanduser().resolve()
if INPUT.is_relative_to(PUBLIC_ROOT) or OUTPUT.is_relative_to(PUBLIC_ROOT):
    raise ValueError('실제 입력과 결과는 공개 저장소 바깥의 private 경로를 사용하세요.')
if OUTPUT.is_relative_to(INPUT) or INPUT.is_relative_to(OUTPUT):
    raise ValueError('입력과 출력 경로는 서로 겹치지 않아야 합니다.')
def private_input(relative_path):
    candidate=(INPUT/relative_path).resolve()
    if not candidate.is_relative_to(INPUT):
        raise ValueError('입력 설정은 private 입력 폴더 내부의 파일만 지정할 수 있습니다.')
    return candidate
CONFIG_PATH=private_input('analysis_config.json')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('private 입력 폴더에 analysis_config.json이 필요합니다.')
CONFIG=json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
FILES=CONFIG['files']; SHEETS=CONFIG['sheets']; PROCESSES=CONFIG['process_names']
ORIGINAL=FILES['original']; SUPPLEMENT=FILES['supplement']
RUN_DIR=OUTPUT/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:6])
required=[CONFIG_PATH]+[private_input(FILES[key]) for key in ['original','supplement','detail_csv','defect_csv','field_mapping_csv']]
required += [private_input(relative) for relative in FILES['source_images']]
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('필수 입력 누락: '+'; '.join(missing))
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def snapshot(): return {str(path.relative_to(INPUT)):sha256(path) for path in sorted(INPUT.rglob('*')) if path.is_file()}
BEFORE=snapshot()
ENV={'python':os.sys.version.split()[0],**{p:metadata.version(p) for p in ['pandas','openpyxl','matplotlib','IPython']}}
display(pd.DataFrame([ENV]))
print('입력 파일 수:',len(BEFORE),'| 경로:',INPUT)

### 시트·컬럼 인벤토리

파일의 저장 범위, 값이 있는 행, 삽입 이미지, 분석용 레코드의 의미를 구분합니다. 문서 설명·헤더·합계행을 생산 표본으로 세지 않습니다. 시트별 원문 범위와 헤더 위치는 private 설정에서 읽으며, 원문 셀과 컬럼명을 별도 결과로 보존합니다.

In [ ]:
inventory=[]; all_cell_data=[]; column_inventory=[]
for file_role in ['original','supplement']:
    filename=FILES[file_role]
    wb=openpyxl.load_workbook(private_input(filename),data_only=True,read_only=False)
    for ws in wb:
        populated=[(i,list(row)) for i,row in enumerate(ws.iter_rows(values_only=True),1) if any(value is not None for value in row)]
        rule=CONFIG['sheet_rules'][file_role][ws.title]
        inventory.append({'파일':filename,'시트':ws.title,'사용범위행수':ws.max_row,'사용범위열수':ws.max_column,
                          '비어있지않은행수':len(populated),'삽입이미지수':len(ws._images),
                          '분석대상셀레코드수':rule['analysis_rows'],'레코드의미':rule['record_meaning'],
                          '원문영역':rule['source_region'],'전체모집단표본수':'확정 불가'})
        header_ranges=rule.get('header_ranges',[f'A1:{openpyxl.utils.get_column_letter(ws.max_column)}1'])
        for header_range in header_ranges:
            for row in ws[header_range]:
                for cell in row:
                    if cell.value is not None:
                        column_inventory.append({'파일':filename,'시트':ws.title,'헤더영역':header_range,
                                                 '헤더셀':cell.coordinate,'원문컬럼명':str(cell.value)})
        for row_no,row in populated:
            for col_no,value in enumerate(row,1):
                if value is not None:
                    all_cell_data.append({'파일':filename,'시트':ws.title,'셀':ws.cell(row_no,col_no).coordinate,
                                          '값':value.isoformat(sep=' ') if isinstance(value,datetime) else value})
    wb.close()
INVENTORY=pd.DataFrame(inventory)
SOURCE_CELLS=pd.DataFrame(all_cell_data)
COLUMN_INVENTORY=pd.DataFrame(column_inventory)
display(INVENTORY)
display(COLUMN_INVENTORY.loc[COLUMN_INVENTORY['시트'].eq(SHEETS['summary'])])

### 원본 테이블과 화면 전사 입력

원본 컬럼명을 유지하고 ID는 문자열로 읽습니다. 합계행과 실제 자재행을 구분합니다. 화면 전사는 별도 CSV와 컬럼 매핑으로 입력하며 출처 파일·시트·이미지·화면 영역을 유지합니다. 정수형 순번이 빈 합계행 때문에 소수형으로 변환되어 키가 달라지지 않도록 원본 셀 형식을 보존합니다.

In [ ]:
def read_exact_table(sheet):
    wb=openpyxl.load_workbook(private_input(ORIGINAL),data_only=True,read_only=True)
    ws=wb[sheet]; values=list(ws.values); header=list(values[0])
    if len(set(header)) != len(header): raise ValueError('중복 헤더: '+sheet)
    # object preserves the original integer-valued sequence cells when footer rows contain blanks.
    # Otherwise pandas would promote integer 1 to float 1.0 and string keys would fail to match.
    df=pd.DataFrame(values[1:],columns=header,dtype=object)
    df['출처행']=range(2,len(df)+2); df['출처파일']=ORIGINAL; df['출처시트']=sheet
    wb.close(); return df
summary=read_exact_table(SHEETS['summary'])
material_all=read_exact_table(SHEETS['material'])
materials=material_all.loc[material_all['지시번호'].notna()].copy()
material_footer=material_all.loc[material_all['지시번호'].isna()].copy()
printing_passes=read_exact_table(SHEETS['printing'])
inventory_stock=read_exact_table(SHEETS['stock'])
detail=pd.read_csv(private_input(FILES['detail_csv']),dtype='string',keep_default_na=False)
defects=pd.read_csv(private_input(FILES['defect_csv']),dtype='string',keep_default_na=False)
field_mapping=pd.read_csv(private_input(FILES['field_mapping_csv']),dtype='string',keep_default_na=False)
for frame in [summary,materials,detail,defects,printing_passes]:
    for column in ['지시번호','작업순서','실적순번','LOT NO','품목코드']:
        if column in frame: frame[column]=frame[column].astype('string')
for frame,columns in [(summary,['지시수량','양품수량','불량수량']),
                      (detail,['지시수량','실적수량','양품수량','불량수량']),
                      (materials,['소요량']),(defects,['불량수량'])]:
    for column in columns: frame[column]=pd.to_numeric(frame[column],errors='raise')
display(Markdown(f'**상세 LOT {len(detail)}행 — 화면 전사**'))
display(detail[['지시번호','작업순서','실적순번','공정명','LOT NO','설비명','실적수량','양품수량','불량수량','시작시간','종료시간']])
display(Markdown(f'**원본 공정 요약 {len(summary)}행 — 상세 LOT와 집계 수준 구분**'))
display(summary[['지시번호','작업순서','공정명','LOT NO','양품수량','불량수량']])
display(field_mapping)

### 누락·중복·기록 시각

정의한 키에서 결측과 반복을 계산합니다. 같은 작업지시의 여러 공정·실적은 반복될 수 있으므로 반복 자체를 업무 오류로 단정하지 않습니다. 시각 범위와 달력상 경과시간을 확인하고, 요약 시각이 모두 같으면 실제 소요/대기시간에 사용하지 않습니다.

In [ ]:
KEY=['지시번호','작업순서','실적순번','LOT NO']
def quality_row(name,frame,keys):
    null=frame[keys].isna().any(axis=1) | frame[keys].eq('').any(axis=1)
    valid=frame.loc[~null,keys]
    return {'테이블':name,'관측행수':len(frame),'검토키':' + '.join(keys),'키누락행수':int(null.sum()),
            '고유키수':len(valid.drop_duplicates()),'중복그룹소속행수':int(valid.duplicated(keep=False).sum()),
            '추가중복행수':int(valid.duplicated(keep='first').sum()),'범위':'제공된 기록 한정; 모집단 품질률 아님'}
QUALITY=pd.DataFrame([
    quality_row('상세 LOT',detail,KEY),
    quality_row('공정 요약',summary,['지시번호','작업순서']),
    quality_row('자재 소요량',materials,['지시번호','작업순서','실적순번','소요량순번']),
    quality_row('불량 유형',defects,['지시번호','작업순서','실적순번','LOT NO','불량코드']),
])
summary_time_equal=bool((pd.to_datetime(summary['시작시간'])==pd.to_datetime(summary['종료시간'])).all())
summary_same_timestamp=int(pd.concat([summary['시작시간'],summary['종료시간']]).nunique())==1
detail['화면시작_파싱']=pd.to_datetime(detail['시작시간'],format='%Y-%m-%d %H:%M',errors='raise')
detail['화면종료_파싱']=pd.to_datetime(detail['종료시간'],format='%Y-%m-%d %H:%M',errors='raise')
detail['기록경과시간_분']=(detail['화면종료_파싱']-detail['화면시작_파싱']).dt.total_seconds()/60
if (detail['기록경과시간_분']<0).any(): raise ValueError('화면 시각의 종료가 시작보다 빠릅니다.')
PERIOD=pd.DataFrame([
    {'대상':f'상세 LOT 화면 {len(detail)}행','시간컬럼':'시작시간/종료시간','시작':str(detail['화면시작_파싱'].min()),'종료':str(detail['화면종료_파싱'].max()),'해석':'관측된 화면 범위; 전체 생산기간 아님'},
    {'대상':f"{SHEETS['printing']} ({len(printing_passes)}행)",'시간컬럼':'실적시작/실적종료','시작':str(printing_passes['실적시작'].min()),'종료':str(printing_passes['실적종료'].max()),'해석':'인쇄 상세 기록 범위; 초/밀리초 원본 유지'},
    {'대상':f"{SHEETS['summary']} ({len(summary)}행)",'시간컬럼':'시작시간/종료시간','시작':str(summary['시작시간'].min()),'종료':str(summary['종료시간'].max()),'해석':'모든 시작/종료가 동일: 실제 소요/대기시간 분석 제외' if summary_same_timestamp else '시각의 업무 의미 확인 후 사용'},
])
MISSING=pd.DataFrame([
    {'항목':'공정요약 설비명 미기재','누락수':int(summary['설비명'].isna().sum()),'분모':len(summary),'의미':'요약 컬럼의 미기재; 상세 설비 부재로 단정 불가'},
    {'항목':'상세LOT 최종검사 불량귀속','누락수':None,'분모':None,'의미':'연결 데이터 미제공; 최종 불량을 상류/원료LOT에 임의 배분하지 않음'},
    {'항목':'재고 오더번호 미기재','누락수':int(inventory_stock['오더번호'].isna().sum()),'분모':len(inventory_stock),'의미':'재고 스냅샷 한정; 본 작업지시 실적과 분리'},
])
display(QUALITY);display(MISSING);display(PERIOD)
print('요약 모든 시작=종료:',summary_time_equal,'| 요약 전체 시각 한 값:',summary_same_timestamp)

### 키 연결 감사

상세 실적을 공정그룹과 LOT 값 두 기준으로 각각 대조합니다. 자재는 `지시번호+작업순서+실적순번`, 불량유형은 상세 실적의 복합키로 연결합니다. 결측 키는 제외하며 행·고유키 기준 매칭률과 연결 관계의 1:1/1:N/N:1/N:N 구조를 보고합니다.

값의 일치는 물리적 LOT 계보 전체의 입증과 구분합니다. 요약에 LOT가 별도 행으로 없을 수 있으며, 수량을 확장 병합하여 중복 합산하지 않습니다.

In [ ]:
def audit_link(name,left,right,keys,meaning):
    def valid_counts(frame):
        ok=frame[keys].notna().all(axis=1) & ~frame[keys].eq('').any(axis=1)
        return ok,Counter(frame.loc[ok,keys].itertuples(index=False,name=None))
    lok,lc=valid_counts(left);rok,rc=valid_counts(right);common=set(lc)&set(rc)
    matched=sum(lc[k] for k in common)
    rel=Counter(('1' if lc[k]==1 else 'N')+':'+('1' if rc[k]==1 else 'N') for k in common)
    return {'연결':name,'키':' + '.join(keys),'왼쪽행수':len(left),'키누락행수':int((~lok).sum()),
            '매칭행수':matched,'행기준매칭률_pct':100*matched/len(left) if len(left) else None,
            '유효키고유수':len(lc),'매칭고유키수':len(common),'고유키매칭률_pct':100*len(common)/len(lc) if lc else None,
            '1대1키수':rel['1:1'],'1대N키수':rel['1:N'],'N대1키수':rel['N:1'],'N대N키수':rel['N:N'],
            '의미':meaning,'범위':'현재 제공행 한정; 전체 공정 이력 복원율 아님'}
MATCHING=pd.DataFrame([
    audit_link('상세LOT → 공정그룹',detail,summary,['지시번호','작업순서'],'상세 실적과 공정 요약의 집계 구조 연결'),
    audit_link('상세LOT → 요약LOT 값 대조',detail,summary,['지시번호','LOT NO'],'요약에 별도 LOT행이 없을 수 있음; 집계 차이를 오류로 단정하지 않음'),
    audit_link('자재행 → 보충 화면의 작업실적내역',materials,detail,['지시번호','작업순서','실적순번'],'자재행과 작업실적내역의 관계; 제품/불량의 원료 귀속은 별도 정보 필요'),
    audit_link('불량유형 → 보충 화면의 작업실적내역',defects,detail,KEY,'해당 실적의 불량 유형 설명이며 같은 수량을 추가 불량으로 합산하지 않음'),
])
display(MATCHING)

### 수량 검산과 표시 비율

설정된 공정의 상세 양품 합과 공정 요약, 실적의 불량과 유형별 합, 자재 합과 원본 합계행, 공정별 불량 합과 지시수량 차이를 비교합니다. 최종 검사 표시 분모는 양품+불량이며, 재검·추가투입·중복계수 정의를 확인해야 합니다. 원료 구성비를 제품별 불량 배분율로 사용하지 않습니다.

In [ ]:
pressure=detail.loc[detail['공정명'].eq(PROCESSES['pressure'])]
printing=detail.loc[detail['공정명'].eq(PROCESSES['printing'])]
final=summary.loc[summary['공정명'].eq(PROCESSES['inspection'])].iloc[0]
inspection_total=int(final['양품수량']+final['불량수량'])
material_sum=float(materials['소요량'].sum())
material_view=materials[['지시번호','작업순서','실적순번','소요량순번','품목코드','품목명','LOT NO','소요량']].copy()
material_view['소요량구성비_pct']=100*material_view['소요량']/material_sum
material_view['소요량단위']='원본 미기재'
checks=[
 {'검산':'대상 공정 상세 양품 합 ↔ 공정 요약 양품','왼쪽':int(pressure['양품수량'].sum()),'오른쪽':int(summary.loc[summary['공정명'].eq(PROCESSES['pressure']),'양품수량'].iloc[0]),'의미':'해당 상세LOT의 집계 일치'},
 {'검산':'선택 공정 상세 불량 ↔ 불량유형 수량','왼쪽':int(printing['불량수량'].sum()),'오른쪽':int(defects['불량수량'].sum()),'의미':'같은 불량수량의 유형 설명; 합산 금지'},
 {'검산':'자재 소요량 합 ↔ 원본 합계행','왼쪽':material_sum,'오른쪽':float(material_footer['소요량'].iloc[0]),'의미':'합계행은 자재 표본에서 제외'},
 {'검산':'공정별 불량 합 ↔ 지시수량−최종양품','왼쪽':int(summary['불량수량'].sum()),'오른쪽':int(summary.iloc[0]['지시수량']-final['양품수량']),'의미':'수량 산술검산; 계보/중복계수 입증 아님'},
]
CHECKS=pd.DataFrame(checks);CHECKS['일치']=CHECKS['왼쪽'].eq(CHECKS['오른쪽'])
RATE=pd.DataFrame([
 {'지표':'선택 공정 관측LOT 불량비율','분자':int(printing['불량수량'].sum()),'분모':int(printing['실적수량'].sum()),'pct':100*printing['불량수량'].sum()/printing['실적수량'].sum(),'한계':'제공된 LOT 관측 한정'},
 {'지표':'최종 검사 표시 불량비율','분자':int(final['불량수량']),'분모':inspection_total,'pct':100*final['불량수량']/inspection_total,'한계':'양품+불량을 표시 분모로 사용; 개별검사/재검 원본 없음'},
 {'지표':'조건부 누적 손실 산술','분자':int(summary['불량수량'].sum()),'분모':int(summary.iloc[0]['지시수량']),'pct':100*summary['불량수량'].sum()/summary.iloc[0]['지시수량'],'한계':'추가투입·재작업·중복계수 없다는 조건 확인 필요'},
])
display(CHECKS);display(RATE);display(material_view);display(defects)
assert CHECKS['일치'].all()

### 시각화

관측된 LOT 수량, 공정 요약 불량, 기록 경과시간을 표시합니다. 축 범위와 표본 수는 입력에서 계산합니다. 경과시간은 가동시간 또는 공정 간 대기시간으로 단정하지 않습니다. 한글 폰트가 없으면 영어 축을 사용하며 `MES_KOREAN_FONT`로 로컬 한글 폰트를 지정할 수 있습니다.

In [ ]:
font_path=os.environ.get('MES_KOREAN_FONT','')
font_name=None
if font_path and Path(font_path).is_file():
    font_manager.fontManager.addfont(font_path);font_name=font_manager.FontProperties(fname=font_path).get_name()
if not font_name:
    installed={font.name for font in font_manager.fontManager.ttflist}
    font_name=next((name for name in ['Noto Sans CJK KR','Malgun Gothic','AppleGothic','NanumGothic'] if name in installed),None)
KOREAN=font_name is not None
if KOREAN: plt.rcParams['font.family']=font_name
plt.rcParams['axes.unicode_minus']=False
FIGURES={}
fig,ax=plt.subplots(figsize=(10,4))
labels=detail['LOT NO'].tolist()
ax.bar(labels,detail['양품수량'],label='양품' if KOREAN else 'Good',color='#347d91')
ax.bar(labels,detail['불량수량'],bottom=detail['양품수량'],label='불량' if KOREAN else 'Defect',color='#bc5545')
quantity_max=max(float(detail['실적수량'].max()),1)
for i,row in detail.iterrows():
    ax.text(i,row['실적수량']+quantity_max*.01,f"{row['실적수량']:,} / {row['불량수량']:,}",ha='center')
ax.set_ylim(0,quantity_max*1.15);ax.legend(frameon=False)
ax.set_title(f'확인된 상세 LOT {len(detail)}행: 전체 / 불량 수량 (서로 다른 공정, 합산 금지)' if KOREAN else f'{len(detail)} observed LOT records: total / defect (do not sum across processes)')
ax.set_ylabel('수량' if KOREAN else 'Count');fig.tight_layout();display(fig);FIGURES['01_detail_lot_quantities.png']=fig;plt.close(fig)

fig,ax=plt.subplots(figsize=(10,4))
process_labels=summary['공정명'].tolist() if KOREAN else [CONFIG['process_labels_en'].get(name,f'Process {i+1}') for i,name in enumerate(summary['공정명'])]
bars=ax.bar(process_labels,summary['불량수량'],color='#bc5545');ax.bar_label(bars,padding=3)
ax.set_ylim(0,max(float(summary['불량수량'].max()),1)*1.2);ax.set_ylabel('불량수량' if KOREAN else 'Defect count')
ax.set_title('원본 공정 요약의 불량수량 — 최종 불량의 상세LOT 귀속은 별도 확인 필요' if KOREAN else 'Process-summary defects; final defects not allocated to detail LOTs')
fig.tight_layout();display(fig);FIGURES['02_process_summary_defects.png']=fig;plt.close(fig)

fig,ax=plt.subplots(figsize=(10,4))
bars=ax.bar(labels,detail['기록경과시간_분']/60,color='#347d91')
elapsed_max=max(float(detail['기록경과시간_분'].max()/60),1)
for bar,minutes in zip(bars,detail['기록경과시간_분']):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+elapsed_max*.01,f'{int(minutes//60)}h {int(minutes%60):02d}m',ha='center')
ax.set_ylim(0,elapsed_max*1.15);ax.set_ylabel('시간' if KOREAN else 'Hours')
ax.set_title('화면 시작~종료의 달력상 경과시간 — 가동시간·공정 대기시간 아님' if KOREAN else 'Recorded elapsed time; not active runtime or inter-process waiting')
fig.tight_layout();display(fig);FIGURES['03_recorded_lot_elapsed_time.png']=fig;plt.close(fig)

### 진행 범위와 추가 확인사항

현재 데이터에 해당하는 상태표와 구체적인 한계는 private 설정에서 읽습니다. LOT 단위로 가능한 확인과 추가 데이터가 필요한 분석을 구분하며, 개별제품 라벨·임의 불량 배분·예측모델·변수 중요도는 생성하지 않습니다.

In [ ]:
STATUS=pd.DataFrame(CONFIG['scope_status'],columns=['요청기능','상태','이번 결과','추가 필요'])
LIMITATIONS=CONFIG['limitations']
display(STATUS)
for text in LIMITATIONS: print('•',text)

### private 결과 저장과 원본 보존

원문 셀·인벤토리·상세/요약/자재/불량 테이블·키 점검·수량 검산·진행 상태를 CSV로, 시각화를 PNG로 저장합니다. 결과 JSON은 private 출력 폴더와 해당 실행 폴더에만 저장합니다. 입력 설정을 포함한 모든 입력의 SHA256을 전후 비교합니다.

In [ ]:
RUN_DIR.mkdir(parents=True,exist_ok=False)
TABLES={'sheet_inventory.csv':INVENTORY,'source_cells.csv':SOURCE_CELLS,'detail_lot_records.csv':detail,
        'process_summary_records.csv':summary,'material_records.csv':materials,'material_footer.csv':material_footer,
        'material_quantity_composition.csv':material_view,'defect_records.csv':defects,'field_mapping.csv':field_mapping,
        'column_inventory.csv':COLUMN_INVENTORY,'quality_checks.csv':QUALITY,'missing_data_checks.csv':MISSING,'recorded_periods.csv':PERIOD,
        'matching_audit.csv':MATCHING,'quantity_reconciliation.csv':CHECKS,'qualified_rates.csv':RATE,'scope_status.csv':STATUS}
for name,frame in TABLES.items():frame.to_csv(RUN_DIR/name,index=False,encoding='utf-8-sig')
for name,fig in FIGURES.items():fig.savefig(RUN_DIR/name,dpi=160,bbox_inches='tight')
AFTER=snapshot();unchanged=BEFORE==AFTER
def records(frame): return json.loads(frame.to_json(orient='records',force_ascii=False,date_format='iso'))
REPORT={
 'product':{column:(values[0] if len(values)==1 else values) for column in ['품목코드','품목명','지시번호'] for values in [summary[column].dropna().unique().tolist()]},
 'detail_lot_records':records(detail),'process_summary_records':records(summary),
 'material_records':records(materials),'defect_records':records(defects),
 'field_mapping':records(field_mapping),'matching_audit':records(MATCHING),'inventory':records(INVENTORY),'column_inventory':records(COLUMN_INVENTORY),
 'quality_checks':records(QUALITY),'missing_data_checks':records(MISSING),'recorded_periods':records(PERIOD),
 'quantity_reconciliation':records(CHECKS),'qualified_rates':records(RATE),'scope_status':records(STATUS),
 'metrics':{'order_count':detail['지시번호'].nunique(),'detail_lot_count':len(detail),'observed_process_group_count':detail['작업순서'].nunique(),
            'summary_process_group_count':len(summary),'printing_good':int(printing['양품수량'].sum()),'printing_defects':int(printing['불량수량'].sum()),
            'pressure_detail_good_sum':int(pressure['양품수량'].sum()),'final_good':int(final['양품수량']),'final_defects':int(final['불량수량']),
            'displayed_final_inspection_total':inspection_total,'displayed_final_defect_rate_pct':100*final['불량수량']/inspection_total,
            'cumulative_displayed_defects':int(summary['불량수량'].sum()),'conditional_cumulative_loss_pct':100*summary['불량수량'].sum()/summary.iloc[0]['지시수량']},
 'limitations':LIMITATIONS,'execution_status':'ALL_CELLS_EXECUTED','run_directory':str(RUN_DIR.relative_to(OUTPUT)),
 'input_sha256_before':BEFORE,'input_sha256_after':AFTER,'originals_unchanged':unchanged,'environment':ENV,
 'chart_font':font_name if KOREAN else 'English fallback',
}
(RUN_DIR/'report_data.json').write_text(json.dumps(REPORT,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT/'report_data.json').write_text(json.dumps(REPORT,ensure_ascii=False,indent=2),encoding='utf-8')
if not unchanged: raise RuntimeError('입력 SHA256이 실행 전후 달라졌습니다. report_data.json을 확인하세요.')
print('입력 SHA256 일치:',unchanged,'| 입력 파일:',len(BEFORE))
print('결과 폴더:',RUN_DIR)
print(f"관측범위: 작업지시 {detail['지시번호'].nunique()}건, 상세LOT {len(detail)}행; 최종 불량의 상류LOT별 귀속 및 예측은 미산출")